# PaperFigures.ipynb

Lightweight paper-ready summaries — no retraining.

**A)** Main-results ΔLL heatmap from `main_results_table.csv`  
**B)** Combined calibration reliability curve + updated `calibration_table.csv`

All outputs saved to the same `results/paper_upgrade/{date}/` directory used by `MathFrameworkExperiments.ipynb`.

In [1]:
import sys, re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# ── Locate repo root (works whether notebook is in "current notebooks/" or root)
REPO_ROOT = Path.cwd()
if REPO_ROOT.name in ("notebooks", "current notebooks") or not (REPO_ROOT / "scripts").is_dir():
    REPO_ROOT = REPO_ROOT.parent

import os
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

from scripts.config import make_config
from scripts.data import load_master_dataset, preprocess_features, build_all_splits
from scripts.bins import compute_X_t, build_all_configs
from scripts.models import StateConditionedNet, StateFreeNet
from scripts.train import build_loaders, load_cached_model
from scripts.eval import compute_ece, compute_brier, _compute_marginal, _build_backoff_matrix
from scripts.plotting import save_fig

# ── Find latest paper_upgrade output directory that has a main_results_table.csv
candidates = sorted(
    Path(REPO_ROOT / "results/paper_upgrade").glob("*/main_results_table.csv"),
    key=lambda p: p.parent.name,  # lexicographic on date string YYYY-MM-DD
)
if not candidates:
    raise FileNotFoundError("No main_results_table.csv found under results/paper_upgrade/*/")

OUT_DIR   = candidates[-1].parent
FIG_DIR   = OUT_DIR / "figures"
CACHE_DIR = OUT_DIR / "cache"
FIG_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cpu")
SEED   = 42
HORIZONS = [1, 2, 5, 10]
N_VALS   = [10, 20, 35, 55]

print(f"Output dir : {OUT_DIR}")
print(f"Fig dir    : {FIG_DIR}")


Output dir : /Users/JanRovirosaIlla/DeepMarkovResearch/results/paper_upgrade/2026-03-07
Fig dir    : /Users/JanRovirosaIlla/DeepMarkovResearch/results/paper_upgrade/2026-03-07/figures


In [2]:
# ══════════════════════════════════════════════════════════
# Part A: Main-results ΔLL heatmap
# ══════════════════════════════════════════════════════════
df_main = pd.read_csv(OUT_DIR / "main_results_table.csv")

# ── delta_ll = test_ll_model − test_ll_marginal is already in the table.
# Verify sign convention: positive delta_ll means model beats marginal.
marginal_rows = df_main[df_main["model"] == "marginal"]
assert (marginal_rows["delta_ll"].abs() < 1e-8).all(), (
    "Expected delta_ll == 0 for marginal rows. Check column definition."
)

def make_heatmap_figure(model_key: str, display_label: str) -> None:
    """Pivot delta_ll to (h × N) and save heatmap + CSV."""
    sub = df_main[df_main["model"] == model_key].copy()
    if len(sub) == 0:
        print(f"  No rows found for model_key={model_key!r} — skipping.")
        return

    pivot = sub.pivot(index="h", columns="N", values="delta_ll")
    pivot = pivot.reindex(index=HORIZONS, columns=N_VALS)

    # Save numeric CSV
    csv_path = OUT_DIR / f"main_results_heatmap_values_{display_label}.csv"
    pivot.to_csv(csv_path)
    print(f"  Saved {csv_path.name}")

    # ── Figure
    vals = pivot.values.astype(float)
    finite_vals = vals[np.isfinite(vals)]
    vmax = max(float(np.abs(finite_vals).max()), 1e-4)

    fig, ax = plt.subplots(figsize=(7, 4.5))
    sns.heatmap(
        pivot,
        ax=ax,
        annot=True,
        fmt=".4f",
        cmap="RdYlGn",
        center=0,
        vmin=-vmax,
        vmax=vmax,
        linewidths=0.5,
        cbar_kws={"label": "ΔLL vs marginal (nats)", "shrink": 0.85},
    )
    ax.set_title(
        f"{display_label}: Δ log-likelihood vs marginal\n"
        f"(positive = better than marginal; rows = horizon h, cols = N)",
        fontsize=12,
    )
    ax.set_xlabel("N (output bins)", fontsize=11)
    ax.set_ylabel("Horizon h", fontsize=11)
    ax.tick_params(axis="both", labelsize=10)
    fig.tight_layout()

    stem = FIG_DIR / f"main_results_heatmap_{display_label}"
    save_fig(fig, stem)
    plt.close(fig)
    print(f"  Saved {stem.name}.pdf + .png")

    print(pivot.to_string())
    print()

print("=== Part A: main-results heatmap ===")
make_heatmap_figure("state_cond_nn", "state_cond")
make_heatmap_figure("state_free_nn", "state_free")
print("Part A done.")


=== Part A: main-results heatmap ===
  Saved main_results_heatmap_values_state_cond.csv


  Saved main_results_heatmap_state_cond.pdf + .png
N         10        20        35        55
h                                         
1  -0.048579 -0.006227  0.002621 -0.032755
2  -0.011472 -0.002195 -0.020552 -0.092629
5  -0.023601 -0.087008 -0.093147 -0.022820
10 -0.009118  0.021921 -0.062181  0.025339

  Saved main_results_heatmap_values_state_free.csv
  Saved main_results_heatmap_state_free.pdf + .png
N         10        20        35        55
h                                         
1  -0.002056  0.003322 -0.000177 -0.040589
2  -0.001308 -0.025035 -0.073578 -0.069896
5  -0.057490 -0.071210 -0.109665 -0.025916
10 -0.025977 -0.006648 -0.010568  0.015738

Part A done.


In [3]:
# ══════════════════════════════════════════════════════════
# Part B setup: load dataset, build configs for (h=1,N=55) and (h=10,N=55)
# ══════════════════════════════════════════════════════════
cfg = make_config()

CALIB_H = [1, 10]
CALIB_N = 55

prices, F_raw, feature_cols = load_master_dataset("dataset")
n_features = F_raw.shape[1]
splits = build_all_splits(prices, CALIB_H)

# Use h=1 train set for z-score normalisation (consistent with main notebook)
idx_train_h1 = splits[1]["idx_train"]
F_normed = preprocess_features(F_raw, idx_train_h1)

train_end_h1 = splits[1]["idx_train"][-1] + 1
X_t_all, N_XT, edges_xt = compute_X_t(prices, cfg.n_xt_target, train_end_h1)
print(f"N_XT={N_XT}, train_end_h1={train_end_h1}")

configs_calib = build_all_configs(
    prices, F_normed, X_t_all,
    CALIB_H, [CALIB_N], N_XT, edges_xt, splits,
    sigma_anchor=cfg.sigma_anchor,
    results_dir=None,
)
print(f"Configs built: {list(configs_calib.keys())}")

# Quick sanity: N_actual for (h, N=55)
for h in CALIB_H:
    c = configs_calib[(h, CALIB_N)]
    print(f"  (h={h}, N=55): N_actual={c['N_actual']}, n_test={len(c['idx_test'])}")


N_XT=55, train_end_h1=1656
Configs built: [(1, 55), (10, 55)]
  (h=1, N=55): N_actual=55, n_test=356
  (h=10, N=55): N_actual=55, n_test=354


In [4]:
# ══════════════════════════════════════════════════════════
# Part B: inference, calibration metrics, reliability curve
# ══════════════════════════════════════════════════════════

def get_probs_from_cache(model_type: str, h: int, N: int) -> tuple:
    """Load cached model weights, run test forward pass, return (probs, y_true).
    Returns (None, None) with a printed message if cache missing.
    """
    c = configs_calib[(h, N)]
    N_actual = c["N_actual"]
    weight_path = CACHE_DIR / f"calib_{model_type}_h{h}_N{N}_seed{SEED}.pt"
    if not weight_path.exists():
        print(f"  [MISSING cache] {weight_path.name}")
        return None, None

    if model_type == "state_cond":
        model = StateConditionedNet(
            n_features, N_XT, N_actual,
            hidden_dims=tuple(cfg.hidden_dims), dropout=cfg.dropout,
        )
    else:
        model = StateFreeNet(
            n_features, N_actual,
            hidden_dims=tuple(cfg.hidden_dims), dropout=cfg.dropout,
        )

    load_cached_model(model, weight_path)
    model.eval()

    _, _, test_loader = build_loaders(
        c, F_normed, X_t_all,
        batch_train=cfg.batch_train, batch_eval=cfg.batch_eval,
    )

    probs_list, y_list = [], []
    with torch.no_grad():
        for F_b, xt_b, y_b in test_loader:
            logits = model(F_b.to(DEVICE), xt_b.to(DEVICE))
            probs_list.append(F.softmax(logits, dim=1).cpu().numpy())
            y_list.append(y_b.numpy())

    probs = np.concatenate(probs_list, axis=0)
    y_true = np.concatenate(y_list, axis=0)
    return probs, y_true


def get_backoff_probs(h: int, N: int) -> tuple:
    """Compute backoff probabilities on the test set (no model loading needed)."""
    c = configs_calib[(h, N)]
    N_actual = c["N_actual"]
    X_tr = X_t_all[c["idx_train"]]
    y_tr = c["y_all"][c["idx_train"]]
    y_te = c["y_all"][c["idx_test"]]
    X_te = X_t_all[c["idx_test"]]

    marginal = _compute_marginal(y_tr, N_actual)

    # Tune alpha/tau on val set
    X_va = X_t_all[c["idx_val"]]
    y_va = c["y_all"][c["idx_val"]]
    from scripts.eval import mean_log_likelihood
    best_ll, best_alpha, best_tau = -np.inf, None, None
    for alpha in cfg.alpha_grid:
        for tau in cfg.tau_grid:
            A, _, _ = _build_backoff_matrix(X_tr, y_tr, N_XT, N_actual, alpha, tau, marginal)
            ll = mean_log_likelihood(A[X_va], y_va)
            if ll > best_ll:
                best_ll, best_alpha, best_tau = ll, alpha, tau
    A_bk, _, _ = _build_backoff_matrix(X_tr, y_tr, N_XT, N_actual,
                                        best_alpha, best_tau, marginal)
    probs_te = A_bk[X_te]  # (n_test, N_actual)
    return probs_te, y_te


# ── Collect calibration data for all 4 key model+horizon combos
MODELS_ORDER = [
    ("state_cond", "state_cond h=1,  N=55", 1,  CALIB_N, "C0", "-"),
    ("state_free",  "state_free  h=1,  N=55", 1,  CALIB_N, "C1", "--"),
    ("state_cond", "state_cond h=10, N=55", 10, CALIB_N, "C2", "-"),
    ("state_free",  "state_free  h=10, N=55", 10, CALIB_N, "C3", "--"),
]

calib_table_rows = []
rel_curves = []   # list of (label, conf_bins, acc_bins, ece, color, ls)

for model_type, display_label, h, N, color, ls in MODELS_ORDER:
    probs, y_true = get_probs_from_cache(model_type, h, N)
    if probs is None:
        continue
    c = configs_calib[(h, N)]
    N_actual = c["N_actual"]
    n_test = len(c["idx_test"])
    event_fn = lambda y, N_=N_actual: y < N_ // 2

    ece, conf_bins, acc_bins = compute_ece(probs, y_true, event_fn, n_bins=10)
    brier = compute_brier(probs, y_true, event_fn)

    calib_table_rows.append({
        "model": model_type, "h": h, "N": N,
        "ece_neg_return": ece, "brier_neg_return": brier, "n_test": n_test,
    })
    rel_curves.append((display_label, conf_bins, acc_bins, ece, color, ls))
    print(f"  {display_label:30s}  ECE={ece:.4f}  Brier={brier:.4f}  n_test={n_test}")

# Add backoff for completeness (optional — included in original calibration_table)
for h in CALIB_H:
    probs_bk, y_true_bk = get_backoff_probs(h, CALIB_N)
    c = configs_calib[(h, CALIB_N)]
    N_actual = c["N_actual"]
    n_test = len(c["idx_test"])
    event_fn = lambda y, N_=N_actual: y < N_ // 2
    ece_bk, _, _ = compute_ece(probs_bk, y_true_bk, event_fn, n_bins=10)
    brier_bk = compute_brier(probs_bk, y_true_bk, event_fn)
    calib_table_rows.append({
        "model": "backoff", "h": h, "N": CALIB_N,
        "ece_neg_return": ece_bk, "brier_neg_return": brier_bk, "n_test": n_test,
    })
    print(f"  backoff h={h}, N=55                  ECE={ece_bk:.4f}  Brier={brier_bk:.4f}")

# ── Save updated calibration_table.csv (with n_test column)
df_calib = pd.DataFrame(calib_table_rows)
df_calib = df_calib.sort_values(["h", "N", "model"]).reset_index(drop=True)
df_calib.to_csv(OUT_DIR / "calibration_table.csv", index=False)
print(f"\nSaved calibration_table.csv ({len(df_calib)} rows)")
print(df_calib.to_string(index=False))

# ── Combined reliability curve (single figure, 4 neural curves + diagonal)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white",
                     "axes.grid": True, "grid.alpha": 0.3})
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot([0, 1], [0, 1], "k--", linewidth=1.2, label="Perfect calibration")

for label, conf_bins, acc_bins, ece, color, ls in rel_curves:
    valid = conf_bins > 0
    ax.plot(conf_bins[valid], acc_bins[valid],
            marker="o", markersize=4, color=color, linestyle=ls,
            label=f"{label}  (ECE={ece:.3f})")

ax.set_xlabel("Mean predicted P(negative return)", fontsize=11)
ax.set_ylabel("Empirical frequency of negative return", fontsize=11)
ax.set_title("Reliability curve — negative return event\n(y < N/2)", fontsize=11)
ax.legend(fontsize=8, loc="upper left")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
fig.tight_layout()
save_fig(fig, FIG_DIR / "reliability_neg_return")
plt.close(fig)
print("\nSaved reliability_neg_return.pdf + .png")


  state_cond h=1,  N=55           ECE=0.0793  Brier=0.2468  n_test=356
  state_free  h=1,  N=55          ECE=0.0948  Brier=0.2510  n_test=356
  state_cond h=10, N=55           ECE=0.1114  Brier=0.2482  n_test=354
  state_free  h=10, N=55          ECE=0.1102  Brier=0.2484  n_test=354
  backoff h=1, N=55                  ECE=0.0808  Brier=0.2485
  backoff h=10, N=55                  ECE=0.1096  Brier=0.2479

Saved calibration_table.csv (6 rows)
     model  h  N  ece_neg_return  brier_neg_return  n_test
   backoff  1 55        0.080825          0.248452     356
state_cond  1 55        0.079298          0.246790     356
state_free  1 55        0.094828          0.250977     356
   backoff 10 55        0.109564          0.247923     354
state_cond 10 55        0.111360          0.248173     354
state_free 10 55        0.110171          0.248427     354



Saved reliability_neg_return.pdf + .png


In [5]:
# ══════════════════════════════════════════════════════════
# Final checklist
# ══════════════════════════════════════════════════════════
print(f"\n{'='*55}")
print(f"Output dir: {OUT_DIR}")
print(f"{'='*55}")

required = [
    ("figures/main_results_heatmap_state_cond.pdf",   FIG_DIR / "main_results_heatmap_state_cond.pdf"),
    ("figures/main_results_heatmap_state_free.pdf",   FIG_DIR / "main_results_heatmap_state_free.pdf"),
    ("main_results_heatmap_values_state_cond.csv",    OUT_DIR / "main_results_heatmap_values_state_cond.csv"),
    ("main_results_heatmap_values_state_free.csv",    OUT_DIR / "main_results_heatmap_values_state_free.csv"),
    ("calibration_table.csv",                         OUT_DIR / "calibration_table.csv"),
    ("figures/reliability_neg_return.pdf",            FIG_DIR / "reliability_neg_return.pdf"),
]

all_ok = True
for name, path in required:
    ok = Path(path).exists() and Path(path).stat().st_size > 0
    sz = Path(path).stat().st_size if Path(path).exists() else 0
    mark = "\u2713" if ok else "\u2717"
    print(f"  {mark} {name}  ({sz:,} B)")
    all_ok = all_ok and ok

if all_ok:
    print("\nAll required outputs present \u2713")
else:
    print("\nSome outputs missing \u2717 — check messages above")



Output dir: /Users/JanRovirosaIlla/DeepMarkovResearch/results/paper_upgrade/2026-03-07
  ✓ figures/main_results_heatmap_state_cond.pdf  (19,122 B)
  ✓ figures/main_results_heatmap_state_free.pdf  (18,906 B)
  ✓ main_results_heatmap_values_state_cond.csv  (338 B)
  ✓ main_results_heatmap_values_state_free.csv  (337 B)
  ✓ calibration_table.csv  (406 B)
  ✓ figures/reliability_neg_return.pdf  (19,270 B)

All required outputs present ✓


In [6]:
# ══════════════════════════════════════════════════════════
# Shared-scale A_t heatmap snapshots (6 files)
# ══════════════════════════════════════════════════════════
import sys
sys.path.insert(0, str(REPO_ROOT))  # ensure updated plotting.py is visible
import importlib, scripts.plotting as _plt_mod
importlib.reload(_plt_mod)
from scripts.plotting import plot_At_heatmap_snapshot, save_fig

CACHE_DIR_SNAPS = Path("results/paper_upgrade/2026-03-07/cache")
FIG_DIR_SNAPS   = Path("results/paper_upgrade/2026-03-07/figures")

SNAP_TIMES  = [208, 253, 297]
SNAP_MODELS = ["state_cond", "state_free"]

# 1. Load all six matrices
matrices = {}
for model_type in SNAP_MODELS:
    A_full = np.load(CACHE_DIR_SNAPS / f"A_t_ck_{model_type}_h1.npy")
    for t in SNAP_TIMES:
        matrices[(model_type, t)] = A_full[t]
        print(f"  Loaded {model_type} t={t}: shape={A_full[t].shape}  "
              f"row-sum range [{A_full[t].sum(axis=1).min():.6f}, {A_full[t].sum(axis=1).max():.6f}]")

# 2. Global vmax
global_vmax = max(A.max() for A in matrices.values())
print(f"\nGlobal vmin=0.0, vmax={global_vmax:.6f}")

# 3. Sanity check: row sums ≈ 1 for one snapshot per model
for model_type in SNAP_MODELS:
    A = matrices[(model_type, SNAP_TIMES[0])]
    row_sums = A.sum(axis=1)
    max_dev = float(np.abs(row_sums - 1.0).max())
    status = "OK" if max_dev < 1e-6 else f"WARN max_dev={max_dev:.2e}"
    print(f"  Row-sum check {model_type} t={SNAP_TIMES[0]}: max deviation={max_dev:.2e}  [{status}]")

# 4. Re-plot with shared scale
print()
for model_type in SNAP_MODELS:
    for t in SNAP_TIMES:
        A = matrices[(model_type, t)]
        fig = plot_At_heatmap_snapshot(
            A, f"t{t}", FIG_DIR_SNAPS,
            model_name=model_type,
            vmin=0.0, vmax=global_vmax,
        )
        plt.close(fig)
        pdf = FIG_DIR_SNAPS / f"At_heatmap_{model_type}_t{t}.pdf"
        png = FIG_DIR_SNAPS / f"At_heatmap_{model_type}_t{t}.png"
        print(f"  {pdf.name}: {pdf.stat().st_size:,} B  |  {png.name}: {png.stat().st_size:,} B")

print(f"\nvmin=0.0  vmax={global_vmax:.6f}")
print("All 6 snapshots regenerated with shared colour scale.")


  Loaded state_cond t=208: shape=(55, 55)  row-sum range [1.000000, 1.000000]
  Loaded state_cond t=253: shape=(55, 55)  row-sum range [1.000000, 1.000000]
  Loaded state_cond t=297: shape=(55, 55)  row-sum range [1.000000, 1.000000]
  Loaded state_free t=208: shape=(55, 55)  row-sum range [1.000000, 1.000000]
  Loaded state_free t=253: shape=(55, 55)  row-sum range [1.000000, 1.000000]
  Loaded state_free t=297: shape=(55, 55)  row-sum range [1.000000, 1.000000]

Global vmin=0.0, vmax=0.030217
  Row-sum check state_cond t=208: max deviation=1.19e-07  [OK]
  Row-sum check state_free t=208: max deviation=0.00e+00  [OK]

  At_heatmap_state_cond_t208.pdf: 25,204 B  |  At_heatmap_state_cond_t208.png: 63,601 B


  At_heatmap_state_cond_t253.pdf: 24,765 B  |  At_heatmap_state_cond_t253.png: 63,364 B
  At_heatmap_state_cond_t297.pdf: 25,331 B  |  At_heatmap_state_cond_t297.png: 63,202 B


  At_heatmap_state_free_t208.pdf: 21,614 B  |  At_heatmap_state_free_t208.png: 51,549 B
  At_heatmap_state_free_t253.pdf: 21,134 B  |  At_heatmap_state_free_t253.png: 51,347 B


  At_heatmap_state_free_t297.pdf: 21,836 B  |  At_heatmap_state_free_t297.png: 51,404 B

vmin=0.0  vmax=0.030217
All 6 snapshots regenerated with shared colour scale.
